# RL Fine-Tuning with Unsloth on AEGIS-Env

**Efficient RL Training for Automated Grading with Small Language Models**

This notebook demonstrates how to fine-tune a small language model (SLM) using **GRPO** (Group Relative Policy Optimization) with **Unsloth** for memory-efficient training on the **AEGIS-Env** grading environment.

### Workflow:
1. ✅ Install dependencies (Unsloth, transformers, TRL, etc.)
2. ✅ Set up the Unsloth environment for GPU acceleration
3. ✅ Configure RL hyperparameters
4. ✅ Load a pre-trained model with LoRA fine-tuning
5. ✅ Define multi-component reward functions
6. ✅ Run the GRPO training loop with curriculum learning
7. ✅ Monitor metrics and learning curves
8. ✅ Save and export the trained model

**Environment**: Google Colab (GPU recommended: T4 or A100)  
**Dataset**: AEGIS-Env with 3 difficulty tiers (easy → medium → hard)

## 1. Install and Import Dependencies

Install all required packages for RL training with Unsloth on AEGIS-Env.

In [ ]:
# %%capture
import os
import importlib.util

# Install core dependencies
!pip install -qqq nest_asyncio

# Check and install PyTorch/CUDA
if importlib.util.find_spec("torch") is None or "COLAB" in "".join(os.environ.keys()):
    try:
        import numpy
        numpy_version = f"numpy=={numpy.__version__}"
    except Exception:
        numpy_version = "numpy"
    
    print("Installing PyTorch with CUDA support...")
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {numpy_version} \
        torchvision bitsandbytes "transformers==4.56.2" trackio

# Install Unsloth
if importlib.util.find_spec("unsloth") is None:
    print("Installing Unsloth...")
    !uv pip install -qqq \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"

# Upgrade core ML packages
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Install OpenEnv and related packages
!uv pip install -qqq "openenv-core[core]>=0.2.2" openai pydantic datasets

In [ ]:
# Core imports
import asyncio
import json
import logging
import os
import sys
from typing import Any, Dict, List, Optional

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ML/RL imports
import torch
import numpy as np
from datasets import Dataset
from transformers import TextStreamer
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

# OpenEnv imports
from client import AegisEnv
from models import AegisAction, AegisObservation
from inference import build_user_prompt, _parse_action, _strip_markdown_json_fence

# Async support
import nest_asyncio
nest_asyncio.apply()

print("✅ All imports successful!")
print(f"🔥 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"📊 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Set Up Unsloth Environment

Configure Unsloth for optimal training performance, device selection, and model loading.

In [ ]:
# Configuration for Unsloth
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 8

logger.info(f"🎯 Device: {DEVICE}")
logger.info(f"📐 Max Sequence Length: {MAX_SEQ_LENGTH}")
logger.info(f"🎛️  LoRA Rank: {LORA_RANK}")

# Environment configuration
AEGIS_OPENENV_BASE = os.environ.get(
    "AEGIS_OPENENV_BASE",
    "https://nishithp2004-aegis-env.hf.space/openenv"
)

MODEL_NAME = os.environ.get(
    "MODEL_NAME",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
)

logger.info(f"🌐 OpenEnv Base: {AEGIS_OPENENV_BASE}")
logger.info(f"🤖 Model: {MODEL_NAME}")

## 3. Initialize RL Training Configuration

Define hyperparameters for GRPO training including learning rate, batch size, and training episodes.

In [ ]:
# RL Training Hyperparameters
TRAINING_CONFIG = {
    # Learning dynamics
    "learning_rate": 2e-4,
    "weight_decay": 0.001,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "linear",
    
    # Batch and gradient configuration
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 2,
    
    # GRPO-specific
    "temperature": 0.8,
    "num_generations": 2,
    
    # Training duration
    "num_train_epochs": 1,  # Train through entire dataset 1 times
    "save_strategy": "epoch",  # Save checkpoint after each epoch
    
    # Model sequence limits
    "max_prompt_length": 2048,
    "max_completion_length": 1024,
    
    # Dataset
    "n_per_tier": 40,  # Samples per difficulty tier
}

# Dataset configuration
DATASET_CONFIG = {
    "seeds": {
        "easy": 1000,
        "medium": 11000,
        "hard": 21000,
    },
    "tiers": ["easy", "medium", "hard"],
}

logger.info("✅ Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    logger.info(f"   {key}: {value}")

## 4. Load Pre-trained Model with Unsloth

Load a small language model with LoRA fine-tuning for efficient training.

In [ ]:
logger.info(f"📥 Loading model: {MODEL_NAME}")

# Load model with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
)

logger.info("✅ Model loaded successfully")

# Apply LoRA for efficient fine-tuning
logger.info(f"🎛️  Applying LoRA (rank={LORA_RANK})...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

logger.info("✅ LoRA applied successfully")
logger.info(f"🤖 Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 5. Build Training Dataset with Curriculum Learning

Fetch training examples from AEGIS-Env across all difficulty tiers with curriculum progression.

In [ ]:
# Helper functions for dataset construction
def _run_async(coro):
    """Run async coroutine in synchronous context."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(coro)


async def replay_to_mentor_obs(env: AegisEnv, seed: int, task_name: str) -> AegisObservation:
    """Replay through pipeline stages to reach mentor stage."""
    obs = (await env.reset(seed=seed, task_name=task_name)).observation
    
    # Deterministic replay through pipeline
    ms = float(obs.max_score or 1.0)
    mid = max(0.0, min(ms, ms * 0.5))
    
    # Arbiter
    a1 = AegisAction(
        proposed_score=mid,
        agent_reasoning="Arbiter: preliminary assessment aligned with the rubric.",
        routing_decision="proceed",
    )
    obs = (await env.step(a1)).observation
    
    # Scrutinizer
    a2 = AegisAction(
        proposed_score=mid,
        agent_reasoning="Scrutinizer: refined criterion-by-criterion review.",
        routing_decision="proceed",
    )
    obs = (await env.step(a2)).observation
    
    # Validator
    a3 = AegisAction(
        proposed_score=mid,
        agent_reasoning="Validator: consistency check passed; proceed to mentor.",
        routing_decision="proceed",
    )
    obs = (await env.step(a3)).observation
    
    if str(getattr(obs, "current_stage", "") or "") != "mentor":
        raise RuntimeError(f"Expected mentor stage, got {getattr(obs, 'current_stage', None)}")
    
    return obs


async def build_one_row(task_name: str, seed: int, base_url: str) -> Dict[str, Any]:
    """Build a single training example."""
    env = AegisEnv(base_url=base_url)
    await env.connect()
    try:
        obs = await replay_to_mentor_obs(env, seed, task_name)
        user_text = build_user_prompt(4, None, 0.0, [], obs)
        return {
            "prompt": [{"role": "user", "content": user_text}],
            "answer": 0,
            "task_name": task_name,
            "episode_seed": int(seed),
            "mentor_max_score": float(obs.max_score or 1.0),
        }
    finally:
        await env.close()


def build_dataset(base_url: str, n_per_tier: int = 40) -> Dataset:
    """Build training dataset with curriculum learning."""
    rows: List[Dict] = []
    tier_seeds = DATASET_CONFIG["seeds"]
    
    logger.info(f"📊 Building dataset with {n_per_tier} samples per tier...")
    
    for task_name in DATASET_CONFIG["tiers"]:
        base_seed = tier_seeds[task_name]
        logger.info(f"  Generating {task_name} tier (base_seed={base_seed})...")
        
        for i in range(n_per_tier):
            seed = base_seed + i
            try:
                row = _run_async(build_one_row(task_name, seed, base_url))
                rows.append(row)
                if (i + 1) % 10 == 0:
                    logger.info(f"    ✓ {task_name}: {i + 1}/{n_per_tier} samples")
            except Exception as e:
                logger.warning(f"    ✗ Failed {task_name} sample {i}: {e}")
    
    logger.info(f"✅ Dataset ready: {len(rows)} total samples")
    return Dataset.from_list(rows)


# Build the dataset
logger.info("🚀 Building training dataset...")
train_ds = build_dataset(AEGIS_OPENENV_BASE, TRAINING_CONFIG["n_per_tier"])
logger.info(f"📈 Dataset size: {len(train_ds)} examples")

## 6. Define Multi-Component Reward Functions

Create reward functions for GRPO training: format validation, output quality, and environment rewards.

In [ ]:
async def mentor_env_reward_one(
    completion_text: str,
    seed: int,
    task_name: str,
    mentor_max_score: float,
    base_url: str,
) -> float:
    """🏆 Evaluate completion against environment mentor stage.
    
    Scale: [-1.0, +1.0]
    - +1.0: Perfect grading (accuracy=1.0, valid format, full flow bank)
    - 0.0: Neutral performance
    - -1.0: Complete failure
    
    Environment already validates format via _coerce_aegis_action(), so this is
    the single source of truth for reward. Returns scaled [-1, +1].
    """
    env = AegisEnv(base_url=base_url)
    await env.connect()
    try:
        obs = await replay_to_mentor_obs(env, seed, task_name)
        ms = float(obs.max_score or mentor_max_score or 1.0)
        
        action = _parse_action(_strip_markdown_json_fence(completion_text), ms)
        sr = await env.step(action)
        r = float(sr.reward if sr.reward is not None else 0.0)
        
        # Scale [0, 1] environment reward to [-1, +1] range
        return 2.0 * r - 1.0
    except Exception as e:
        logger.debug(f"Mentor reward error: {e}")
        return -1.0  # Error penalty
    finally:
        await env.close()


def mentor_success(
    prompts,
    completions,
    episode_seed,
    task_name,
    mentor_max_score,
    base_url: str,
    **kwargs,
) -> List[float]:
    """🌟 Single source of truth: environment-based evaluation.
    
    No redundant format checks—the environment validates all actions via
    _coerce_aegis_action() internally. This function only evaluates grading quality.
    """
    scores = []
    for c, seed, task, ms in zip(completions, episode_seed, task_name, mentor_max_score):
        text = c[0]["content"]
        try:
            coro = mentor_env_reward_one(text, int(seed), str(task), float(ms), base_url)
            score = float(_run_async(coro))
            scores.append(score)
        except Exception as e:
            logger.debug(f"Mentor evaluation failed: {e}")
            scores.append(-1.0)
    return scores


logger.info("✅ Reward function defined:")
logger.info("   • mentor_success: Environment-based evaluation → [-1.0, +1.0]")
logger.info("\n💡 Note: Format validation is handled by the environment's _coerce_aegis_action()")
logger.info("         so we use a single source of truth for the reward signal.")

## 7. Configure and Initialize GRPO Trainer

Set up GRPO training configuration and initialize the trainer with all components.

In [ ]:
# Compute prompt length
sample_prompt = train_ds[0]["prompt"]
sample_text = tokenizer.apply_chat_template(
    sample_prompt, tokenize=False, add_generation_prompt=True
)
max_prompt_length = len(tokenizer.encode(sample_text))
logger.info(f"📏 Sample prompt length: {max_prompt_length} tokens")

# Adjust sequence lengths
max_seq_length = MAX_SEQ_LENGTH
max_prompt_length = min(max_prompt_length + 8, max_seq_length // 2)
max_completion_length = min(
    TRAINING_CONFIG["max_completion_length"],
    max_seq_length - max_prompt_length
)

logger.info(f"📐 Max prompt length: {max_prompt_length} tokens")
logger.info(f"📐 Max completion length: {max_completion_length} tokens")

# Configure GRPO training
training_args = GRPOConfig(
    # Learning dynamics
    temperature=TRAINING_CONFIG["temperature"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
    optim="adamw_8bit",
    
    # Logging and saving
    logging_steps=1,
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # GRPO specifics
    num_generations=TRAINING_CONFIG["num_generations"],
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    
    # Training duration
    num_train_epochs=TRAINING_CONFIG['num_train_epochs'],
    save_strategy=TRAINING_CONFIG['save_strategy'],
    save_total_limit=5,
    
    # Output
    output_dir="outputs_grpo_aegis",
    logging_dir="outputs_grpo_aegis/logs",
    
    # Mixed precision
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8,
)

# Wrap mentor_success with base_url
def mentor_success_with_url(prompts, completions, episode_seed, task_name, mentor_max_score, **kwargs):
    return mentor_success(
        prompts,
        completions,
        episode_seed,
        task_name,
        mentor_max_score,
        base_url=AEGIS_OPENENV_BASE,
        **kwargs,
    )

# Initialize GRPO trainer
logger.info("🎯 Initializing GRPO Trainer...")
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[mentor_success_with_url],
    args=training_args,
    train_dataset=train_ds,
)

logger.info("✅ Trainer initialized and ready for training!")

## 8. Run GRPO Training Loop

Execute the main training loop with curriculum learning from easy to hard tasks.

In [ ]:
logger.info("=" * 80)
logger.info("🚀 STARTING GRPO TRAINING")
logger.info("=" * 80)
logger.info(f"Training epochs: {TRAINING_CONFIG['num_train_epochs']}")
logger.info(f"Total samples: {len(train_ds)} (easy + medium + hard)")
logger.info(f"Save strategy: {TRAINING_CONFIG['save_strategy']} (checkpoint after each epoch)")
logger.info(f"Reward function: mentor_success (environment-based evaluation)")
logger.info("=" * 80)

# Run training
trainer.train()

logger.info("=" * 80)
logger.info("✅ TRAINING COMPLETE!")
logger.info("=" * 80)

## 9. Monitor Training Metrics

Track and visualize training progress with real-time monitoring.

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Load training logs
log_dir = Path("outputs_grpo_aegis/logs")
if log_dir.exists():
    logger.info("📊 Loading training metrics...")
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("GRPO Training Metrics", fontsize=16, fontweight="bold")
    
    # Initialize metric lists
    steps = []
    losses = []
    rewards = []
    learning_rates = []
    
    # Try to load from trainer state
    if hasattr(trainer, "state"):
        training_state = trainer.state
        if hasattr(training_state, "log_history"):
            for log_entry in training_state.log_history:
                if "step" in log_entry:
                    steps.append(log_entry["step"])
                if "loss" in log_entry:
                    losses.append(log_entry["loss"])
                if "reward" in log_entry:
                    rewards.append(log_entry["reward"])
                if "learning_rate" in log_entry:
                    learning_rates.append(log_entry["learning_rate"])
    
    # Plot metrics
    ax = axes[0, 0]
    if losses:
        ax.plot(steps[:len(losses)], losses, marker="o", label="Loss")
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("Training Loss")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "Loss data not available", ha="center", va="center")
        ax.set_title("Training Loss")
    
    ax = axes[0, 1]
    if rewards:
        ax.plot(steps[:len(rewards)], rewards, marker="s", color="green", label="Reward")
        ax.set_xlabel("Step")
        ax.set_ylabel("Average Reward")
        ax.set_title("Episode Rewards")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "Reward data not available", ha="center", va="center")
        ax.set_title("Episode Rewards")
    
    ax = axes[1, 0]
    if learning_rates:
        ax.plot(steps[:len(learning_rates)], learning_rates, marker="^", color="orange")
        ax.set_xlabel("Step")
        ax.set_ylabel("Learning Rate")
        ax.set_title("Learning Rate Schedule")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "Learning rate data not available", ha="center", va="center")
        ax.set_title("Learning Rate Schedule")
    
    ax = axes[1, 1]
    ax.text(0.05, 0.95, f"Total Steps: {TRAINING_CONFIG['max_steps']}", 
            ha="left", va="top", fontsize=11, family="monospace")
    ax.text(0.05, 0.85, f"Learning Rate: {TRAINING_CONFIG['learning_rate']}", 
            ha="left", va="top", fontsize=11, family="monospace")
    ax.text(0.05, 0.75, f"Batch Size: {TRAINING_CONFIG['per_device_train_batch_size']}", 
            ha="left", va="top", fontsize=11, family="monospace")
    ax.text(0.05, 0.65, f"Gradient Accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}", 
            ha="left", va="top", fontsize=11, family="monospace")
    ax.text(0.05, 0.55, f"Temperature: {TRAINING_CONFIG['temperature']}", 
            ha="left", va="top", fontsize=11, family="monospace")
    ax.text(0.05, 0.45, f"Model: {MODEL_NAME.split('/')[-1]}", 
            ha="left", va="top", fontsize=11, family="monospace")
    ax.axis("off")
    ax.set_title("Training Configuration")
    
    plt.tight_layout()
    plt.savefig("outputs_grpo_aegis/training_metrics.png", dpi=100, bbox_inches="tight")
    plt.show()
    
    logger.info("✅ Training metrics visualized and saved")
else:
    logger.info("⚠️  Log directory not found. Metrics will be available after training.")

## 10. Save and Export Trained Model

Save the fine-tuned model checkpoint and export for inference.

In [ ]:
output_dir = Path("outputs_grpo_aegis")
output_dir.mkdir(parents=True, exist_ok=True)

logger.info(f"💾 Saving fine-tuned model to {output_dir}...")

# Save model weights and config
model.save_pretrained(str(output_dir / "final_model"))
tokenizer.save_pretrained(str(output_dir / "final_model"))

logger.info("✅ Model saved successfully")

# Save training summary
summary = {
    "model_name": MODEL_NAME,
    "training_config": TRAINING_CONFIG,
    "dataset_config": DATASET_CONFIG,
    "total_samples": len(train_ds),
    "output_dir": str(output_dir),
    "model_path": str(output_dir / "final_model"),
}

summary_path = output_dir / "training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

logger.info(f"📋 Training summary saved to {summary_path}")

# Prepare inference example
logger.info("\n🎯 Model is ready for inference!")
logger.info(f"   Model path: {output_dir / 'final_model'}")
logger.info(f"   Tokenizer: {output_dir / 'final_model'}")
logger.info("\n📚 Usage example:")
logger.info(f"""
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="{output_dir / 'final_model'}",
    load_in_4bit=True,
    max_seq_length=4096,
)

prompt = "Your prompt here"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512)
response = tokenizer.decode(outputs[0])
""")

logger.info("\n" + "=" * 80)
logger.info("🎉 RL TRAINING PIPELINE COMPLETE!")
logger.info("=" * 80)
logger.info(f"Final model location: {output_dir / 'final_model'}")
logger.info(f"Training logs: {output_dir / 'logs'}")
logger.info(f"Checkpoints: {output_dir / 'checkpoint-*'}")
logger.info("=" * 80)

# Push to Hugging Face Hub
logger.info("\n🌐 Pushing model to Hugging Face Hub...")

# Extract model name for the repo naming
model_shortname = MODEL_NAME.split("/")[-1]
hf_repo_name = f"NishithP2004/aegis-{model_shortname}"

logger.info(f"📤 Target repo: {hf_repo_name}")

try:
    # Push model to hub
    model.push_to_hub(
        repo_id=hf_repo_name,
        token=os.environ.get("HF_TOKEN"),
        private=False,
        commit_message=f"GRPO fine-tuned model for AEGIS-Env - {TRAINING_CONFIG['max_steps']} steps",
    )
    
    # Push tokenizer to hub
    tokenizer.push_to_hub(
        repo_id=hf_repo_name,
        token=os.environ.get("HF_TOKEN"),
        private=False,
    )
    
    logger.info(f"✅ Model successfully pushed to {hf_repo_name}")
    logger.info(f"🔗 URL: https://huggingface.co/{hf_repo_name}")
    
except Exception as e:
    logger.warning(f"⚠️  Failed to push to HF Hub: {e}")
    logger.info("   Make sure HF_TOKEN environment variable is set")
    logger.info("   You can set it with: !huggingface-cli login")

## Appendix: Quick Reference & Troubleshooting

### 🎯 Quick Hyperparameter Tuning

Adjust these in Section 3 to customize training:

- **Learning Rate**: Higher for faster learning, lower for stability (default: 2e-4)
- **Batch Size**: Increase for more stable gradients (default: 1 per device)
- **Temperature**: Controls generation diversity (default: 0.8)
- **Max Steps**: Total training iterations (default: 200)
- **LoRA Rank**: Higher rank = more parameters but slower (default: 8)

### 🐛 Common Issues

| Issue | Solution |
|-------|----------|
| CUDA Out of Memory | Reduce batch size, reduce max_seq_length, or use larger GPU instance |
| Dataset Building Fails | Check AEGIS_OPENENV_BASE URL is accessible from Colab |
| Slow Training | Enable mixed precision (already configured), use gradient accumulation |
| Import Errors | Restart kernel and re-run dependency installation cell |

### 📊 Expected Performance

With 120 samples (40 per tier) and 200 steps:
- Training time: ~8-12 hours on T4 GPU
- Final model size: ~3GB (4-bit quantized)
- Expected final reward: 0.5-0.7 range after convergence

### 🔗 Useful Links

- **GitHub Repo**: https://github.com/NishithP2004/aegis_env
- **HF Space**: https://huggingface.co/spaces/NishithP2004/aegis-env
- **Unsloth Docs**: https://github.com/unslothai/unsloth
- **TRL Docs**: https://huggingface.co/docs/trl/

### 💡 Next Steps

After training completes:
1. Download the model: `final_model/` directory
2. Push to Hugging Face Hub: `model.push_to_hub("your-hub-name")`
3. Evaluate on test set from AEGIS-Env
4. Fine-tune further with harder examples or different reward weights

## Optional: Inference & Evaluation

Test the trained model on AEGIS-Env (run after training completes).

In [ ]:
# Load the fine-tuned model
logger.info("📥 Loading fine-tuned model for inference...")

final_model_path = "outputs_grpo_aegis/final_model"

model_infer, tokenizer_infer = FastLanguageModel.from_pretrained(
    model_name=final_model_path,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
)

# Prepare model for inference
FastLanguageModel.for_inference(model_infer)

logger.info("✅ Model loaded for inference")

# Test inference on a sample
if len(train_ds) > 0:
    logger.info("\n🧪 Running test inference on a training sample...")
    
    sample = train_ds[0]
    prompt_text = tokenizer_infer.apply_chat_template(
        sample["prompt"],
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Generate response
    inputs = tokenizer_infer(prompt_text, return_tensors="pt").to("cuda")
    outputs = model_infer.generate(
        **inputs,
        temperature=0.7,
        max_new_tokens=512,
    )
    
    response = tokenizer_infer.decode(outputs[0], skip_special_tokens=True)
    
    logger.info("\n💬 Model Output:")
    logger.info("-" * 80)
    # Extract just the response part (after the prompt)
    response_only = response.split("Assistant:\n")[-1].strip() if "Assistant:" in response else response
    logger.info(response_only[:500] + "..." if len(response_only) > 500 else response_only)
    logger.info("-" * 80)
    
    # Try to parse as JSON
    try:
        parsed = _parse_action(_strip_markdown_json_fence(response_only), sample["mentor_max_score"])
        logger.info(f"\n✅ Successfully parsed as AegisAction:")
        logger.info(f"   - Proposed Score: {parsed.proposed_score}")
        logger.info(f"   - Reasoning: {parsed.agent_reasoning[:100]}...")
        logger.info(f"   - Routing Decision: {parsed.routing_decision}")
    except Exception as e:
        logger.warning(f"⚠️  Failed to parse as JSON: {e}")

logger.info("\n" + "=" * 80)
logger.info("✨ Inference test complete!")
logger.info("=" * 80)